# CEG-WM Content V4 — clean-null whitening fit initial user-only GPU handoff

This Colab notebook performs the frozen **initial invocation only** of the 32-clean-image whitening fit. It checks out branch `stage-a-content-adaptive-dual-branch-v4-whitened-lf` at exact `e86f9fbb0817011ef0ecf9d4518215e08b2c9033`, installs only that checkout's declared package, mounts the user's Drive, reads `HF_TOKEN` from Colab Secrets, and invokes the existing fit runner exactly once.

Frozen identities are fit protocol `cegwm-stage-a-content-v4-clean-null-whitening-fit-v1` with e86-bound digest `746ae77c661d42ffd622a925f335396b58270b21846fb0871a4dff7df3996a80`, run `content-v4-whitening-fit-746ae77c661d-e86f9fbb0817`, asset role `content_v4_clean_null_whitening_operator_v1`, current manifest SHA-256 `0766827406d0107693102bc823a76bbfa3f76815f43475daad4530a259faaed7`, and Archive-source manifest SHA-256 `5d7388a92c98aa5fb1996369bae8de65360e2d25fa7569400135753257bb6e86`.

Create the Colab Secret `HF_TOKEN` before starting. Run Sections 1–3 once and in order. Section 4 is runner-free and may be used after completion, failure, interruption, or reconnect to validate and download an already existing exact asset pair. Do not retry, resume, substitute, or modify this handoff after any failure, interruption, OOM, Drive issue, or runtime loss. This produces an engineering asset only: it has scientific denominator zero and makes no score, Gate, threshold, calibration, fixed-FPR, method-validity, or promotion claim.

## 1. Fresh checkout and frozen identity proof

Run this before installation, Secret access, Drive mounting, or GPU/model work. The frozen source path must not already exist, and the fresh clone must match the named source branch, execution exact, and clean state.

In [ ]:
import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-adaptive-dual-branch-v4-whitened-lf"
EXACT = "e86f9fbb0817011ef0ecf9d4518215e08b2c9033"
RUNNER_MODULE = "experiments.run_content_v4_clean_null_whitening_fit"
FIT_PROTOCOL_ID = "cegwm-stage-a-content-v4-clean-null-whitening-fit-v1"
FIT_PROTOCOL_DIGEST = "746ae77c661d42ffd622a925f335396b58270b21846fb0871a4dff7df3996a80"
RUN_ID = "content-v4-whitening-fit-746ae77c661d-e86f9fbb0817"
ASSET_ROLE_ID = "content_v4_clean_null_whitening_operator_v1"
ASSET_FILENAME = ASSET_ROLE_ID + ".json"
MANIFEST_SHA256 = "0766827406d0107693102bc823a76bbfa3f76815f43475daad4530a259faaed7"
ARCHIVE_MANIFEST_SHA256 = "5d7388a92c98aa5fb1996369bae8de65360e2d25fa7569400135753257bb6e86"
RECEIPT_PREFIX = "CEGWM_CONTENT_V4_WHITENING_RECEIPT"
FAILURE_PREFIX = "CEGWM_CONTENT_V4_WHITENING_HANDOFF_FAILURE"

repo = pathlib.Path("/content/cegwm-stage-a-content-v4-whitening-fit-source")
local_root = pathlib.Path("/content/cegwm-stage-a-content-v4-whitening-fit-local")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v4_whitening_fit")
bound_result_dir = artifact_sink / RUN_ID / EXACT
asset_path = bound_result_dir / ASSET_FILENAME
checksum_path = bound_result_dir / (ASSET_FILENAME + ".sha256")

HALTED = False
_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "ValueError",
}

def stop(stage, error_class="RuntimeError"):
    global HALTED
    if HALTED:
        return
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    HALTED = True
    print(FAILURE_PREFIX + " " + json.dumps(
        {
            "status": "operational_failure",
            "run_id": RUN_ID,
            "producer_exact": EXACT,
            "stage": stage,
            "error_class": error_class,
        },
        sort_keys=True, separators=(",", ":"),
    ), flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

clone = None
try:
    if repo.exists():
        raise FileExistsError
    clone = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError
    clone = None
    if (
        git("branch", "--show-current") != BRANCH
        or git("rev-parse", "HEAD") != EXACT
        or git("status", "--porcelain") != ""
    ):
        raise RuntimeError
except BaseException as error:
    clone = None
    stop("source_checkout_identity_validation", type(error).__name__)

## 2. Install the checked-out project

This installs the package and requirements declared by the frozen checkout exactly once, then proves that the branch, exact, and clean state remain unchanged. Do not change versions or retry if installation fails.

In [ ]:
if not HALTED:
    install = None
    try:
        install = subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            capture_output=True, text=True,
        )
        if install.returncode != 0:
            raise RuntimeError
        install = None
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
        ):
            raise RuntimeError
    except BaseException as error:
        install = None
        stop("dependency_install_and_source_recheck", type(error).__name__)

## 3. Mount Drive, validate the initial-only destinations, and invoke the fit runner once

Create only the Colab Secret `HF_TOKEN`. It is passed only to the single child runner and cleared afterward. The frozen local root is used only for this task's Hugging Face cache. The local root, bound Drive result directory, asset JSON, and SHA sidecar must all be absent before invocation. Raw child stdout is captured but never emitted, and raw stderr is discarded. This cell prints exactly one validated bounded success receipt or one non-raising sanitized failure line.

In [ ]:
import os
import re
from google.colab import drive, userdata

if not HALTED:
    runner_env = None
    completed = None
    runner_stdout = ""
    runner_rc = None
    runner_exception_class = None
    hf_token = ""
    try:
        drive.mount("/content/drive")
        if (
            local_root.exists()
            or bound_result_dir.exists()
            or asset_path.exists()
            or checksum_path.exists()
        ):
            raise FileExistsError
        local_root.mkdir(parents=True, exist_ok=False)
        hf_cache = local_root / "hf-cache"
        hf_cache.mkdir(parents=False, exist_ok=False)
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_result_dir.exists()
            or asset_path.exists()
            or checksum_path.exists()
        ):
            raise RuntimeError

        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["HF_TOKEN"] = hf_token
        runner_env["HF_HOME"] = str(hf_cache)
        hf_token = ""
        completed = subprocess.run(
            [
                sys.executable,
                "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
                "--artifact-sink", str(artifact_sink),
            ],
            cwd=repo,
            env=runner_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
        )
        runner_rc = completed.returncode
        runner_stdout = completed.stdout
    except BaseException as error:
        runner_exception_class = type(error).__name__
    finally:
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("HF_TOKEN", None)
            runner_env.pop("HF_HOME", None)
        runner_env = None

    if runner_exception_class is not None:
        stop("fit_runner_launch", runner_exception_class)
    elif completed is None or runner_rc != 0:
        stop("fit_runner_nonzero_return", "CalledProcessError")
    if HALTED:
        runner_stdout = ""
        completed = None

if not HALTED:
    validated_line = ""
    lines = []
    line = ""
    receipt = None
    try:
        lines = runner_stdout.splitlines()
        if len(lines) != 1:
            raise RuntimeError
        line = lines[0]
        if not line.startswith(RECEIPT_PREFIX + " ") or len(line) > 4096:
            raise RuntimeError
        receipt = json.loads(line.split(" ", 1)[1])
        expected_fields = {
            "asset_digest", "asset_role_id", "fit_protocol_digest",
            "producer_exact", "run_id", "unit_count",
        }
        if not isinstance(receipt, dict) or set(receipt) != expected_fields:
            raise RuntimeError
        if (
            re.fullmatch(r"[0-9a-f]{64}", receipt["asset_digest"]) is None
            or receipt["asset_role_id"] != ASSET_ROLE_ID
            or receipt["fit_protocol_digest"] != FIT_PROTOCOL_DIGEST
            or receipt["producer_exact"] != EXACT
            or receipt["run_id"] != RUN_ID
            or receipt["unit_count"] != 32
        ):
            raise RuntimeError
        validated_line = line
    except BaseException as error:
        stop("fit_runner_receipt_validation", type(error).__name__)
    finally:
        runner_stdout = ""
        completed = None
        lines = []
        line = ""
        receipt = None

    if not HALTED:
        HALTED = True
        print(validated_line, flush=True)

## 4. Read-only validation and download of the existing asset pair

This self-contained cell never invokes or resumes the runner. It may be run after success, failure, interruption, or reconnect. It discovers only the exact bound JSON/SHA pair written natively by the runner, validates identities and the sidecar hash in memory without recreating or rewriting either file, and downloads both existing files.

In [ ]:
import hashlib
import json
import pathlib
import re
from google.colab import drive, files

EXACT = "e86f9fbb0817011ef0ecf9d4518215e08b2c9033"
FIT_PROTOCOL_ID = "cegwm-stage-a-content-v4-clean-null-whitening-fit-v1"
FIT_PROTOCOL_DIGEST = "746ae77c661d42ffd622a925f335396b58270b21846fb0871a4dff7df3996a80"
RUN_ID = "content-v4-whitening-fit-746ae77c661d-e86f9fbb0817"
ASSET_ROLE_ID = "content_v4_clean_null_whitening_operator_v1"
ASSET_FILENAME = ASSET_ROLE_ID + ".json"
MANIFEST_SHA256 = "0766827406d0107693102bc823a76bbfa3f76815f43475daad4530a259faaed7"
ARCHIVE_MANIFEST_SHA256 = "5d7388a92c98aa5fb1996369bae8de65360e2d25fa7569400135753257bb6e86"
FAILURE_PREFIX = "CEGWM_CONTENT_V4_WHITENING_HANDOFF_FAILURE"
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v4_whitening_fit")
result_dir = artifact_sink / RUN_ID / EXACT
asset_path = result_dir / ASSET_FILENAME
checksum_path = result_dir / (ASSET_FILENAME + ".sha256")
ARTIFACT_HALTED = False

def artifact_stop(stage):
    global ARTIFACT_HALTED
    if ARTIFACT_HALTED:
        return
    ARTIFACT_HALTED = True
    print(FAILURE_PREFIX + " " + json.dumps(
        {
            "status": "operational_failure",
            "run_id": RUN_ID,
            "producer_exact": EXACT,
            "stage": stage,
            "error_class": "RuntimeError",
        },
        sort_keys=True, separators=(",", ":"),
    ), flush=True)

try:
    if not pathlib.Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    expected_names = [ASSET_FILENAME, ASSET_FILENAME + ".sha256"]
    if (
        not result_dir.is_dir()
        or not asset_path.is_file()
        or not checksum_path.is_file()
        or sorted(path.name for path in result_dir.iterdir()) != sorted(expected_names)
    ):
        raise RuntimeError
    asset_bytes = asset_path.read_bytes()
    checksum_fields = checksum_path.read_text(encoding="ascii").strip().split()
    if (
        len(checksum_fields) != 2
        or re.fullmatch(r"[0-9a-f]{64}", checksum_fields[0]) is None
        or checksum_fields[1] != ASSET_FILENAME
        or hashlib.sha256(asset_bytes).hexdigest() != checksum_fields[0]
    ):
        raise RuntimeError
    asset = json.loads(asset_bytes)
    words = asset.get("whitening_words_be_hex_channel_major_band_minor")
    contract = asset.get("fit_contract")
    manifest = contract.get("fit_manifest") if isinstance(contract, dict) else None
    if (
        not isinstance(asset, dict)
        or asset.get("asset_role_id") != ASSET_ROLE_ID
        or asset.get("fit_protocol_id") != FIT_PROTOCOL_ID
        or asset.get("fit_protocol_digest") != FIT_PROTOCOL_DIGEST
        or asset.get("run_id") != RUN_ID
        or asset.get("producer_exact") != EXACT
        or not isinstance(contract, dict)
        or contract.get("scientific_denominator") != 0
        or not isinstance(manifest, dict)
        or manifest.get("raw_sha256") != MANIFEST_SHA256
        or manifest.get("archive_source_raw_sha256") != ARCHIVE_MANIFEST_SHA256
        or not isinstance(words, list)
        or len(words) != 96
        or any(not isinstance(word, str) or re.fullmatch(r"[0-9a-f]{8}", word) is None for word in words)
        or b'"prompt":' in asset_bytes
    ):
        raise RuntimeError
except BaseException:
    asset_bytes = b""
    asset = None
    words = contract = manifest = None
    artifact_stop("existing_asset_pair_validation")

if not ARTIFACT_HALTED:
    try:
        files.download(str(asset_path))
        files.download(str(checksum_path))
        asset_bytes = b""
        asset = None
        words = contract = manifest = None
    except BaseException:
        artifact_stop("existing_asset_pair_download")

## Return and stop boundary

- On success, return the single validated `CEGWM_CONTENT_V4_WHITENING_RECEIPT` line and the downloaded `content_v4_clean_null_whitening_operator_v1.json` plus `.sha256` sidecar.
- On handoff or artifact failure, return only the single sanitized `CEGWM_CONTENT_V4_WHITENING_HANDOFF_FAILURE` line. Never expose raw child stdout, stderr, traceback, Secrets, prompts, observations, latents, energies, model state, or private state.
- Never relaunch, retry, resume, refit, reconstruct, replace, or fall back after success, failure, interruption, OOM, dependency/model/backend error, missing artifact, Drive issue, or Colab loss.
- Do not change the branch, exact, roots, model, VAE, processor, observation contract, manifest, ordered fit bindings, detrend, DCT, rings, ridge, serialization, or destination.
- The returned W is an engineering public asset from a 32-unit clean-null fit with scientific denominator zero. This notebook does not build or run the reserved formal Content V4 protocol and makes no keyed-attribution, score, Gate, threshold, calibration, fixed-FPR, robustness, mechanism, Stage-A, main, paper, publication, or promotion claim.